# RunPod Serverless, From Zero to a Production API

The notebook behind the demo: sign up → API key → **this notebook** → pay-per-second GPU inference.

What we prove, in order:
1. An idle endpoint keeps **zero workers** — nothing to pay when nothing runs
2. The **first request from zero** (cold start: queue + worker spin-up)
3. A **warm request** — the same call once a worker is active
4. A **burst** of concurrent requests — the queue absorbing load
5. **The bill** — what this whole session actually cost

## 1. Setup

Config lives in `.env` (`RUNPOD_API_KEY`, `ENDPOINT_ID`, `GPU_HOURLY_USD`).
The endpoint can be one you deployed yourself (any Hub template) or a Runpod-hosted public model endpoint — the API is identical.

If you need to create your own endpoint first:
`uv run scripts/create_endpoint.py --template-id <HUB_TEMPLATE_ID>`

In [1]:
import json
import os
import time

import httpx
from dotenv import load_dotenv

load_dotenv('../.env')  # adjust to '../.env' if run from notebooks/

API_KEY = os.environ['RUNPOD_API_KEY']
ENDPOINT_ID = os.environ['ENDPOINT_ID']
GPU_HOURLY_USD = float(os.getenv('GPU_HOURLY_USD', '0.69'))

BASE_URL = f'https://api.runpod.ai/v2/{ENDPOINT_ID}'
HEADERS = {'Authorization': f'Bearer {API_KEY}', 'Content-Type': 'application/json'}
PROMPT = 'A small red cube on a white background'
print(f'Endpoint: {BASE_URL}')

Endpoint: https://api.runpod.ai/v2/black-forest-labs-flux-1-schnell


## 2. Health check — how many workers are running right now?

Min workers = 0 means the endpoint **scales to zero** when idle — this check is where we confirm there's nothing running and nothing to pay for.

(Public model endpoints don't expose `/health`; your own endpoints do.)

In [2]:
with httpx.Client(timeout=30.0) as client:
    resp = client.get(f'{BASE_URL}/health', headers=HEADERS)

if resp.status_code == 401:
    print('/health not available on this endpoint (normal for public model endpoints).')
else:
    resp.raise_for_status()
    print(json.dumps(resp.json(), indent=2))

/health not available on this endpoint (normal for public model endpoints).


## 3. Cold start — the first request from zero workers

Submit an async job (`POST /run`), then poll `GET /status/{id}`. The first call pays the cold start: worker provisioning + model load. `delayTime` in the response is exactly that cost, in milliseconds.

In [3]:
with httpx.Client() as client:
    t0 = time.monotonic()
    resp = client.post(
        f'{BASE_URL}/run', headers=HEADERS,
        json={'input': {'prompt': PROMPT}}, timeout=30.0,
    )
    resp.raise_for_status()
    job_id = resp.json()['id']
    print(f'job submitted: {job_id}')

    last = None
    while True:
        job = client.get(f'{BASE_URL}/status/{job_id}', headers=HEADERS, timeout=30.0).json()
        if job['status'] != last:
            print(f"t={time.monotonic() - t0:6.1f}s  status -> {job['status']}")
            last = job['status']
        if job['status'] == 'COMPLETED':
            cold = job
            break
        assert job['status'] not in ('FAILED', 'CANCELLED'), job
        time.sleep(2)

print(f"cold start delayTime: {cold['delayTime']} ms, executionTime: {cold['executionTime']} ms")

from IPython.display import Image as IPyImage, display

if isinstance(cold.get('output'), dict) and isinstance(cold['output'].get('result'), str):
    display(IPyImage(url=cold['output']['result']))
else:
    print('Output:', json.dumps(cold['output'])[:300])

job submitted: 5169c354-3ee4-420c-a74a-51e1405c4474-e2


t=   0.7s  status -> IN_QUEUE


t=   5.2s  status -> IN_PROGRESS


t=  16.6s  status -> COMPLETED
cold start delayTime: 3877 ms, executionTime: 11914 ms


## 4. Warm request — the FlashBoot difference

Same call, immediately after, via `/runsync` (synchronous — waits for the result). With a worker already active and FlashBoot enabled, `delayTime` should collapse from seconds to milliseconds.

In [4]:
with httpx.Client() as client:
    t0 = time.monotonic()
    resp = client.post(
        f'{BASE_URL}/runsync?wait=120000', headers=HEADERS,
        json={'input': {'prompt': PROMPT}}, timeout=150.0,
    )
    resp.raise_for_status()
    warm = resp.json()
    warm_wall = time.monotonic() - t0

print(f"warm delayTime:    {warm['delayTime']:>6} ms")
print(f"cold delayTime:    {cold['delayTime']:>6} ms")
print(f"difference:        {cold['delayTime'] - warm['delayTime']:>6} ms faster when warm")
print()
print('Output:', json.dumps(warm['output'])[:200])
print()
if isinstance(warm.get('output'), dict) and isinstance(warm['output'].get('result'), str):
    display(IPyImage(url=warm['output']['result']))

warm delayTime:       481 ms
cold delayTime:      3877 ms
difference:          3396 ms faster when warm

Output: {"cost": 0.003, "result": "https://image.runpod.ai/wavespeed-flux-schnell/1cc2e7d3f6cc43b0bd825fb275fe9c66/result.jpeg"}



## 5. Burst — 20 concurrent requests

Fire 20 requests at once and watch how the platform absorbs them: each lands in the queue, workers pick jobs up as they scale, and every request still completes.

In [5]:
import asyncio

TOPICS = [
    'GPU virtualization', 'quantization of LLMs', 'KV caching', 'speculative decoding',
    'LoRA adapters', 'mixture-of-experts', 'flash attention', 'gradient checkpointing',
    'RAG pipelines', 'vector databases', 'RLHF', 'distillation',
    'batch inference', 'CUDA streams', 'tensor parallelism', 'pipeline parallelism',
    'activation checkpointing', 'model sharding', 'token healing', 'continuous batching',
]

async def one(client, prompt):
    t = time.monotonic()
    r = await client.post(
        f'{BASE_URL}/runsync?wait=300000', headers=HEADERS,
        json={'input': {'prompt': prompt}}, timeout=320.0,
    )
    r.raise_for_status()
    job = r.json()
    job['_wall_s'] = time.monotonic() - t
    return job

t0 = time.monotonic()
async with httpx.AsyncClient() as client:
    results = await asyncio.gather(*(one(client, f'An icon of {t}') for t in TOPICS))
burst_wall = time.monotonic() - t0

completed = [j for j in results if j['status'] == 'COMPLETED']
walls = sorted(j['_wall_s'] for j in completed)
delays = sorted(j['delayTime'] for j in completed)
print(f'completed:            {len(completed)} / {len(results)}')
print(f'burst wall time:      {burst_wall:.1f}s')
print(f'request wall median:  {walls[len(walls)//2]:.1f}s   max: {walls[-1]:.1f}s')
print(f'delayTime median:     {delays[len(delays)//2]} ms')

completed:            20 / 20
burst wall time:      93.3s
request wall median:  51.9s   max: 93.2s
delayTime median:     35825 ms


## 6. The bill

Per-second billing, from worker start to full stop. Some endpoints report the exact billed cost per job in the output — when they don't, we estimate from worker time × the GPU's hourly rate.

In [6]:
jobs = [cold, warm] + results

reported = [
    j['output']['cost'] for j in jobs
    if isinstance(j.get('output'), dict) and j['output'].get('cost') is not None
]
if reported:
    print(f'Billed cost reported by the endpoint: ${sum(reported):.5f}')
else:
    worker_s = sum(j.get('delayTime', 0) + j.get('executionTime', 0) for j in jobs) / 1000.0
    print(f'Worker time: {worker_s:.1f}s at ${GPU_HOURLY_USD}/hr '
          f'= ${worker_s * GPU_HOURLY_USD / 3600.0:.5f}')
print()
print('That is the entire economics of the demo: a real GPU, real concurrency, pennies.')

Billed cost reported by the endpoint: $0.06600

That is the entire economics of the demo: a real GPU, real concurrency, pennies.


## Wrap-up

**Runpod is the AI Developer Cloud. Sign Up Today.** — link in the video description.

Reproduce this end to end:
1. Sign up and create an API key (console → Settings → API Keys — use a *Restricted* key)
2. Deploy an endpoint: console → Serverless → New Endpoint, or `scripts/create_endpoint.py`
3. Put the key and endpoint ID in `.env`
4. Run this notebook top to bottom